In [1]:

%pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import sys
print(sys.executable)

c:\Users\nisas\AppData\Local\Programs\Python\Python312\python.exe


In [ ]:
# Cell 1: Import Libraries
import pandas as pd
import numpy as np
import ast
import pickle
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Cell 2: Load Data
movies = pd.read_csv(r'C:\Users\nisas\OneDrive\Documents\GitHub\Movie-Recommender\archive')
credits = pd.read_csv(r'C:\Users\nisas\OneDrive\Documents\GitHub\Movie-Recommender\archive')

# Cell 3: Merge Datasets
movies = movies.merge(credits, on='title')

# Cell 4: Select Important Features
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
movies.dropna(inplace=True)

# Cell 5: Extract Genres
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

movies['genres'] = movies['genres'].apply(convert)

# Cell 6: Extract Keywords
movies['keywords'] = movies['keywords'].apply(convert)

# Cell 7: Extract Cast (Top 3 actors)
def convert_cast(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
            counter += 1
    return L

movies['cast'] = movies['cast'].apply(convert_cast)

# Cell 8: Extract Director
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

movies['crew'] = movies['crew'].apply(fetch_director)

# Cell 9: Convert Overview to List
movies['overview'] = movies['overview'].apply(lambda x: x.split())

# Cell 10: Remove Spaces (so "Sam Worthington" → "SamWorthington")
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ", ""))
    return L1

movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

# Cell 11: Combine All Features into "tags"
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

# Cell 12: Create Final Dataframe
new_df = movies[['movie_id', 'title', 'tags']].copy()
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

# Cell 13: Vectorize Using CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

# Cell 14: Calculate Cosine Similarity
similarity = cosine_similarity(vectors)

# Reset index to ensure position matches similarity matrix
new_df = new_df.reset_index(drop=True)

# Cell 15: Test the Recommender (Optional)
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = sorted(list(enumerate(similarity[movie_index])), reverse=True, key=lambda x: x[1])
    
    print(f"\nMovies similar to '{movie}':")
    for i in distances[1:6]:
        print(f"  - {new_df.iloc[i[0]].title}")

recommend('Avatar')

# Cell 16: Save Model Files
pickle.dump(new_df, open('movie_list.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
# Re-check the shapes
print(f"✅ Model saved successfully!")
print(f"Movies shape: {new_df.shape}")
print(f"Similarity shape: {similarity.shape}")

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\nisas\\OneDrive\\Documents\\GitHub\\Movie-Recommender'

In [ ]:
# Show first 10 movies to see exact titles
print("First 10 movies in dataset:")
print(movies['title'].head(10))

# Check if "John Carter" is in the list (exact match)
print("\nIs 'John Carter' in the dataset?")
print('John Carter' in movies['title'].values)

# Show similar titles (case-insensitive, partial match)
print("\nMovies with 'John' in title:")
print(movies[movies['title'].str.contains('John', case=False)] ['title'].tolist())

First 10 movies in dataset:


NameError: name 'movies' is not defined

In [ ]:
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.
